### 1. Save model tas series at obs locations:
1. Load subdaily data for each simulation
2. Gaussian blur each timestep
3. Calculate/store time series at each obs location

In [2]:
from Montreal_UHI_toolbox import *
import dask_image.ndfilters as ndfilters

In [ ]:
station_set = obs
# 1. Load subdaily data for each simulation
subdaily = {}
subdaily['tas_C'],subdaily['tas_T'] = get_outputs('tas')
subdaily['tas_C'],subdaily['tas_T'] = get_outputs('tas')

# Can also load daily for each sim
daily = {}
daily['tasmin_C'],daily['tasmin_T'] = get_outputs('tasmin')
daily['tasmax_C'],daily['tasmax_T'] = get_outputs('tasmax')

In [ ]:
# 2. Gaussian blur each timestep of the full gridded set (ie no temporal blurring)

# Using the conventions from gaussian_blur_xarray
#sigma = (0.,1.5,1.5) # tuple (sigma_temporal,sigma_y,sigma_x)
blurred = {}

# Creating function to leverage dask
def gaussian(da,sigma=(0.,1.5,1.5)):
    return ndfilters.gaussian_filter(da,sigma=sigma)

# Using gaussian inside custom xarray ufunc  
for key in subdaily.keys():
    blurred[key] = xr.apply_ufunc(
        gaussian,
        subdaily[key],
        input_core_dims=[['time','rlat','rlon']],
        output_core_dims=[['time','rlat','rlon']],
        dask='allowed'
    )

# Gratuitous but I'd like to separate the daily and subdaily
for key in daily.keys():
    blurred[key] = xr.apply_ufunc(
        gaussian,
        daily[key],
        input_core_dims=[['time','rlat','rlon']],
        output_core_dims=[['time','rlat','rlon']],
        dask='allowed'
    )

In [27]:
# 3. Time series at each station_set location

series = {} # the dataset(s) to save

for station_set,urban_or_rural in zip([obs_urban,obs_rural],['urban','rural']):
    series[urban_or_rural] = {}
    # Comparisons have to be made in rotated pole to extract time series data
    station_set_rotated_points = rotated_pole.transform_points(ccrs.PlateCarree(), station_set['lon'].values, station_set['lat'].values)
    station_set_rlon = station_set_rotated_points[:, 0]
    station_set_rlat = station_set_rotated_points[:, 1]

    # Interpolate station_set locations, grab time series from blurred fields
    for key in subdaily.keys():
        # Select nearest point to station from blurred sim data
        series[urban_or_rural][key] = blurred[key].sel(rlat=xr.DataArray(station_set_rlat, dims='points'),rlon=xr.DataArray(station_set_rlon, dims='points'),method='nearest')

    # Still gratuitous
    for key in daily.keys():
        # Select nearest point to station from blurred sim data
        series[urban_or_rural][key] = blurred[key].sel(rlat=xr.DataArray(station_set_rlat, dims='points'),rlon=xr.DataArray(station_set_rlon, dims='points'),method='nearest')


In [13]:
key = 'tas_C'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [14]:
key = 'tas_C'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [15]:
key = 'tas_T'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [16]:
key = 'tas_T'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [28]:
key = 'tasmin_C'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [29]:
key = 'tasmin_C'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [30]:
key = 'tasmax_C'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [31]:
key = 'tasmax_C'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [32]:
key = 'tasmin_T'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [33]:
key = 'tasmin_T'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [34]:
key = 'tasmax_T'
urban_or_rural = 'urban'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')

In [35]:
key = 'tasmax_T'
urban_or_rural = 'rural'
series[urban_or_rural][key].to_netcdf(f'/runoff/gulley/St_Laurent/intermediates/sim/series_at_obs_locations/{key}_{urban_or_rural}.nc')